In [ ]:
# CELL 1 — Install Dependencies
%pip install -q \
    langchain langchain-core langchain-community langchain-text-splitters \
    langchain-ollama langchain-huggingface langchain-chroma \
    langgraph langgraph-prebuilt langgraph-checkpoint-sqlite \
    sentence-transformers chromadb pypdf \
    aiosqlite \
    prometheus-client prometheus-fastapi-instrumentator \
    pandas numpy openpyxl pyarrow \
    matplotlib seaborn plotly \
    python-dotenv sqlglot

print('Dependencies installed')

In [ ]:
!pip show langchain langchain-core langgraph langchain-ollama langchain-huggingface | grep -E 'Name|Version'

In [ ]:
# CELL 2 — Configure
import os
import json
import urllib.request
from pathlib import Path
from dotenv import load_dotenv

# 1. Load .env
load_dotenv()

# 2. Directory setup
BASE_DIR = Path('./data')
(BASE_DIR / 'attendance').mkdir(parents=True, exist_ok=True)
(BASE_DIR / 'dashboards').mkdir(parents=True, exist_ok=True)

# Add project root to sys.path so cells can import src/
import sys
sys.path.insert(0, str(Path('.').resolve()))

# 3. Model / embedding config
#    EMBED_MODEL uses HuggingFace SentenceTransformers — no Ollama pull needed.
#    The model is downloaded automatically from HuggingFace on first use.
MODEL           = os.getenv('MODEL',           'qwen2.5:14b')
EMBED_MODEL     = os.getenv('EMBED_MODEL',     'BAAI/bge-small-en-v1.5')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_API_KEY  = os.getenv('OLLAMA_API_KEY',  '')

os.environ['MODEL']           = MODEL
os.environ['EMBED_MODEL']     = EMBED_MODEL
os.environ['OLLAMA_BASE_URL'] = OLLAMA_BASE_URL
os.environ['OLLAMA_API_KEY']  = OLLAMA_API_KEY

# 4. Verify Ollama reachability (supports both local and api.ollama.com)
_tags_url = f"{OLLAMA_BASE_URL.rstrip('/')}/api/tags"
_headers  = {'Authorization': f'Bearer {OLLAMA_API_KEY}'} if OLLAMA_API_KEY else {}
try:
    req = urllib.request.Request(_tags_url, headers=_headers)
    with urllib.request.urlopen(req, timeout=5) as r:
        pulled = {m['name'].split(':')[0] for m in json.loads(r.read()).get('models', [])}
    model_base = MODEL.split(':')[0]
    if model_base not in pulled:
        print(f'Warning: {MODEL} may not be available. Run: ollama pull {MODEL}')
    else:
        print(f'Ollama : {OLLAMA_BASE_URL}  (OK — {model_base} ready)')
except Exception:
    print(f'Warning: Ollama not reachable at {OLLAMA_BASE_URL}')
    print('         Chat queries will fail until Ollama is available.')

print(f'\nConfiguration complete')
print(f'  Model       : {MODEL}')
print(f'  Embed model : {EMBED_MODEL}  (HuggingFace, auto-downloaded)')
print(f'  Ollama      : {OLLAMA_BASE_URL}')

## Docker SQL Server (education_db + finance_db)

Run the cells below when you have the Docker container running and the databases seeded.

```bash
# Terminal — start container and seed (one-time, ~5–10 min at large scale)
docker compose -f docker/docker-compose.yml up -d sqlserver
pip install faker                                        # dev dependency — seeder only
python scripts/seed_test_db.py --scale large             # ~5M rows across 34 tables

# Optional: inject low-attendance classes for threshold variation
python scripts/seed_low_attendance.py
# Adds 15,000 rows for sections 9-A (50%), 10-C (58%), 11-E (63%),
# 12-B (68%), 10-D (72%) — all below the 75% threshold.
```

In [ ]:
# CELL 2b — Connect to Docker SQL Server
# Requires: docker compose -f docker/docker-compose.yml up -d sqlserver
#           python scripts/seed_test_db.py --scale large
#
# Skip this cell to stay with the synthetic AttendanceDataStore (no SQL Server needed).

import os

os.environ.update({
    'SQL_SERVER':        'localhost,1433',
    'SQL_DATABASES':     'education_db,finance_db',
    'SQL_PRIMARY_DB':    'education_db',
    'SQL_USERNAME':      'sa',
    'SQL_PASSWORD':      'ExcelsisTest_2024!',
    'SQL_DRIVER':        '{ODBC Driver 18 for SQL Server}',
    'SQL_POOL_SIZE':     '5',
    'SQL_QUERY_TIMEOUT': '30',
    'PRIMARY_TABLE':      'attendance',
    'METRIC_COLUMN':      'status',
    'POSITIVE_VALUE':     'present',
    'DATE_COLUMN':        'date',
    'ENTITY_COLUMN':      'student_id',
    'ENTITY_NAME_COLUMN': 'student_name',
    'GROUP_COLUMNS':      'class_section,grade',
})

from src.sql_store import SQLDataStore

sql_store = SQLDataStore()
print('SQLDataStore connected')
print(f"  Databases : {os.environ['SQL_DATABASES']}")

summary = sql_store.summary()
print(f'\neducation_db / attendance summary:')
for k, v in summary.items():
    print(f'  {k:<26} {v}')

In [ ]:
# CELL 3 — In-Memory Data Store (no SQL Server required)
import uuid
import json
import os
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd

BASE_PATH       = Path('data')
ATTENDANCE_PATH = BASE_PATH / 'attendance'
DASHBOARD_PATH  = BASE_PATH / 'dashboards'

try:
    ATTENDANCE_PATH.mkdir(parents=True, exist_ok=True)
    DASHBOARD_PATH.mkdir(parents=True, exist_ok=True)
    print(f'Data directories ready at: {BASE_PATH.absolute()}')
except PermissionError:
    print('Permission Error: Ensure you have write access to this folder.')

VALID_STATUSES = {'present', 'absent', 'late', 'excused'}


def parse_attendance_query(question: str) -> dict:
    q = question.lower()
    chart_type = 'bar'
    if 'line' in q:   chart_type = 'line'
    elif 'pie' in q:  chart_type = 'pie'
    elif 'heat' in q: chart_type = 'heatmap'

    group_by = 'class'
    if 'week' in q:       group_by = 'week'
    elif 'month' in q:    group_by = 'month'
    elif 'day' in q:      group_by = 'day_of_week'
    elif 'student' in q:  group_by = 'student_id'
    elif 'grade' in q:    group_by = 'grade'

    period = 'all'
    if 'last 7' in q or 'this week' in q:     period = 'last_7_days'
    elif 'last 30' in q or 'this month' in q:  period = 'last_30_days'

    threshold = 75.0
    for word in q.split():
        try:
            val = float(word.strip('%'))
            if 40 < val < 100:
                threshold = val
        except ValueError:
            pass

    use_at_risk = any(
        w in q for w in ['at risk', 'at-risk', 'below', 'missing', 'absent', 'low attendance']
    )
    return {
        'group_by': group_by, 'period': period,
        'chart_type': chart_type, 'threshold': threshold,
        'use_at_risk': use_at_risk,
    }


class AttendanceDataStore:
    def __init__(self, data_path=None):
        self._datasets = {}
        if data_path is not None:
            self._load_from_path(Path(data_path))

    def _load_from_path(self, root: Path) -> None:
        root = Path(root)
        if not root.exists(): return
        paths = ([root] if root.is_file()
                 else sorted(list(root.glob('*.csv')) + list(root.glob('*.xlsx'))
                             + list(root.glob('*.parquet'))))
        for p in paths:
            if not p.is_file(): continue
            suf = p.suffix.lower()
            try:
                if suf == '.csv':            raw = pd.read_csv(p)
                elif suf in ('.xlsx','.xls'): raw = pd.read_excel(p)
                elif suf == '.parquet':       raw = pd.read_parquet(p)
                else: continue
                self.ingest_df(raw, name=p.stem)
            except Exception as ex:
                print(f'Could not load {p}: {ex}')

    def ingest_df(self, df: pd.DataFrame, name: str = 'uploaded') -> dict:
        did = str(uuid.uuid4())
        df  = df.copy()
        df.columns    = [c.strip().lower().replace(' ', '_') for c in df.columns]
        df['date']    = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
        df['status']  = df['status'].str.strip().str.lower()
        df = df[df['status'].isin(VALID_STATUSES)].copy()
        df['is_present']  = df['status'] == 'present'
        df['is_absent']   = df['status'] == 'absent'
        df['is_late']     = df['status'] == 'late'
        df['week']        = df['date'].dt.to_period('W').astype(str)
        df['month']       = df['date'].dt.to_period('M').astype(str)
        df['day_of_week'] = df['date'].dt.day_name()
        self._datasets[did] = {'name': name, 'df': df}
        print(f"Ingested '{name}': {len(df):,} rows")
        return {'dataset_id': did, 'rows': len(df), 'columns': list(df.columns)}

    def merged(self) -> pd.DataFrame:
        if not self._datasets: return pd.DataFrame()
        return pd.concat([v['df'] for v in self._datasets.values()], ignore_index=True)

    def get_at_risk(self, threshold: float = 75.0, grade: str = 'all') -> pd.DataFrame:
        df = self.merged()
        if df.empty: return pd.DataFrame()
        if grade != 'all' and 'class' in df.columns:
            df = df[df['class'].str.upper() == grade.upper()]
        agg_cols = {
            'total':   ('status',     'count'),
            'present': ('is_present', 'sum'),
            'absent':  ('is_absent',  'sum'),
            'late':    ('is_late',    'sum'),
        }
        if 'student_name' in df.columns: agg_cols['name'] = ('student_name', 'first')
        if 'class' in df.columns:        agg_cols['cls']  = ('class', 'first')
        agg = df.groupby('student_id').agg(**agg_cols).reset_index()
        agg['attendance_rate'] = (agg['present'] / agg['total'] * 100).round(1)
        return agg[agg['attendance_rate'] < threshold].sort_values('attendance_rate')

    def compute_stats(self, group_by: str = 'class', period: str = 'all') -> pd.DataFrame:
        df = self.merged()
        if df.empty: return pd.DataFrame()
        if period in ('last_7_days', 'this_week'):
            df = df[df['date'] >= datetime.utcnow() - timedelta(days=7)]
        elif period in ('last_30_days', 'this_month'):
            df = df[df['date'] >= datetime.utcnow() - timedelta(days=30)]
        col = group_by if group_by in df.columns else ('class' if 'class' in df.columns else 'student_id')
        g = df.groupby(col).agg(
            total=('status', 'count'), present=('is_present', 'sum'),
            absent=('is_absent', 'sum'), late=('is_late', 'sum'),
        ).reset_index()
        g['attendance_rate'] = (g['present'] / g['total'] * 100).round(1)
        return g

    def summary(self) -> dict:
        df = self.merged()
        if df.empty: return {'status': 'no_data'}
        return {
            'total_records':           len(df),
            'unique_students':         int(df['student_id'].nunique()),
            'date_range':              {'from': str(df['date'].min().date()), 'to': str(df['date'].max().date())},
            'overall_attendance_rate': round(float(df['is_present'].mean() * 100), 1),
            'total_absences':          int(df['is_absent'].sum()),
            'classes':                 df['class'].unique().tolist() if 'class' in df.columns else [],
        }

    def query_to_df(self, query: str) -> pd.DataFrame:
        p = parse_attendance_query(query)
        return self.get_at_risk(p['threshold']) if p['use_at_risk'] else self.compute_stats(p['group_by'], p['period'])


store = AttendanceDataStore()
print('In-memory data store ready')

In [ ]:
# CELL 4 — Dashboard Builder
import uuid
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

COLORS = {
    'primary': '#1a3c5e', 'success': '#00a86b',
    'warning': '#f5a623', 'danger':  '#e74c3c', 'accent': '#3498db',
}


def build_dashboard(chart_type: str, df: pd.DataFrame, title: str, group_col: str) -> go.Figure:
    avg = df['attendance_rate'].mean() if 'attendance_rate' in df.columns else 0

    if chart_type == 'bar':
        fig = px.bar(
            df.sort_values('attendance_rate'), x=group_col, y='attendance_rate',
            color='attendance_rate', color_continuous_scale=['#e74c3c', '#f5a623', '#00a86b'],
            range_color=[50, 100], title=title, text='attendance_rate',
        )
        fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
        fig.add_hline(y=avg, line_dash='dash', line_color=COLORS['accent'],
                      annotation_text=f'Average: {avg:.1f}%')
        fig.add_hline(y=75,  line_dash='dot',  line_color=COLORS['danger'],
                      annotation_text='At-Risk Threshold (75%)')
    elif chart_type == 'line':
        fig = px.line(df.sort_values(group_col), x=group_col, y='attendance_rate',
                      title=title, markers=True)
        fig.add_hline(y=75, line_dash='dot', line_color=COLORS['danger'],
                      annotation_text='At-Risk (75%)')
    elif chart_type == 'pie':
        totals = {'Present': int(df['present'].sum()), 'Absent': int(df['absent'].sum()),
                  'Late': int(df['late'].sum())}
        fig = px.pie(values=list(totals.values()), names=list(totals.keys()), title=title, hole=0.4,
                     color_discrete_map={'Present': COLORS['success'], 'Absent': COLORS['danger'],
                                         'Late': COLORS['warning']})
    elif chart_type == 'heatmap':
        if 'day_of_week' in df.columns:
            pivot = df.pivot_table(values='attendance_rate', index=group_col, columns='day_of_week')
            fig = go.Figure(go.Heatmap(
                z=pivot.values, x=pivot.columns.tolist(), y=pivot.index.tolist(),
                colorscale=[[0,'#e74c3c'],[0.5,'#f5a623'],[1,'#00a86b']],
                zmin=50, zmax=100, text=pivot.values.round(1), texttemplate='%{text}%',
            ))
            fig.update_layout(title=title)
        else:
            fig = build_dashboard('bar', df, title, group_col)
    else:
        fig = build_dashboard('bar', df, title, group_col)

    fig.update_layout(
        template='plotly_white',
        font=dict(family='Inter, sans-serif', size=13, color=COLORS['primary']),
        title_font=dict(size=18, color=COLORS['primary']),
        margin=dict(t=80, b=60, l=60, r=40),
    )
    return fig


def show_dashboard(fig: go.Figure, save: bool = True):
    fig.show()
    if save:
        _dash = Path('data') / 'dashboards'
        _dash.mkdir(parents=True, exist_ok=True)
        path = str(_dash / f'{uuid.uuid4().hex[:8]}.html')
        fig.write_html(path, include_plotlyjs='cdn')
        print(f'Saved: {path}')


def _mpl_modern_theme():
    sns.set_theme(
        context='notebook', style='whitegrid',
        rc={
            'axes.facecolor':'#f8fafc', 'figure.facecolor':'#eef2f7',
            'grid.color':'#e2e8f0', 'axes.edgecolor':'#cbd5e1',
            'axes.labelcolor': COLORS['primary'], 'text.color': COLORS['primary'],
            'xtick.color': COLORS['primary'], 'ytick.color': COLORS['primary'],
            'font.family':'sans-serif',
            'font.sans-serif':['Inter','DejaVu Sans','Helvetica Neue','Arial','sans-serif'],
            'figure.dpi':120,
        },
    )


def build_modern_static_dashboard(store, period: str = 'all', save: bool = True):
    _mpl_modern_theme()
    summ = store.summary()
    if summ.get('status') == 'no_data':
        fig, ax = plt.subplots(figsize=(9, 2.5))
        ax.text(0.5, 0.5, 'No attendance data loaded.', ha='center', va='center',
                fontsize=14, color=COLORS['primary'])
        ax.set_facecolor('#f8fafc'); ax.axis('off')
        plt.tight_layout()
        return fig

    df_class = store.compute_stats('class', period)
    df_week  = store.compute_stats('week',  period)
    if not df_week.empty:
        wk_col  = 'week' if 'week' in df_week.columns else df_week.columns[0]
        df_week = df_week.sort_values(wk_col)
    df_dow = store.compute_stats('day_of_week', period)
    day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    if not df_dow.empty and 'day_of_week' in df_dow.columns:
        df_dow = df_dow.copy()
        df_dow['day_of_week'] = pd.Categorical(df_dow['day_of_week'], categories=day_order, ordered=True)
        df_dow = df_dow.sort_values('day_of_week')

    rate = summ.get('overall_attendance_rate', 0)
    dr   = summ.get('date_range', {})

    fig = plt.figure(figsize=(18, 13))
    gs  = fig.add_gridspec(nrows=4, ncols=4, height_ratios=[0.11,0.16,1.0,1.0], hspace=0.42, wspace=0.28)

    ax_head = fig.add_subplot(gs[0, :])
    ax_head.set_facecolor(COLORS['primary']); ax_head.axis('off')
    ax_head.text(0.03, 0.5, 'Excelsis 360 — Attendance', color='white',
                 fontsize=16, fontweight='600', va='center', transform=ax_head.transAxes)
    ax_head.text(0.97, 0.5, f"{dr.get('from','—')}  →  {dr.get('to','—')}",
                 color='#cbd5e1', fontsize=11, ha='right', va='center', transform=ax_head.transAxes)

    kpis = [
        ('Attendance rate', f'{rate}%',                          COLORS['success']),
        ('Records',  f"{summ.get('total_records',0):,}",         COLORS['accent']),
        ('Students', f"{summ.get('unique_students',0):,}",       '#38bdf8'),
        ('Absences', f"{summ.get('total_absences',0):,}",        COLORS['danger']),
    ]
    for i, (lab, val, c) in enumerate(kpis):
        ax_k = fig.add_subplot(gs[1, i])
        ax_k.set_facecolor('#f1f5f9')
        for s in ax_k.spines.values(): s.set_visible(False)
        ax_k.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
        ax_k.text(0.5, 0.62, lab, ha='center', va='center', fontsize=10,
                  color='#64748b', transform=ax_k.transAxes)
        ax_k.text(0.5, 0.30, val, ha='center', va='center', fontsize=15,
                  fontweight='700', color=c, transform=ax_k.transAxes)

    ax1 = fig.add_subplot(gs[2, :2])
    if not df_class.empty:
        gcol   = 'class' if 'class' in df_class.columns else df_class.columns[0]
        plot_c = df_class.sort_values('attendance_rate')
        sns.barplot(data=plot_c, y=gcol, x='attendance_rate', palette='crest',
                    hue=gcol, dodge=False, legend=False, ax=ax1)
        ax1.axvline(75,   color=COLORS['danger'], linestyle='--', linewidth=1, alpha=0.85, label='At-risk (75%)')
        ax1.axvline(rate, color='#fbbf24', linestyle='-.', linewidth=1.2, alpha=0.95, label=f'Avg ({rate:.1f}%)')
        ax1.set_xlim(max(45, plot_c['attendance_rate'].min() - 5), 100)
        ax1.set_title('By class', loc='left', fontsize=12, fontweight='600', color=COLORS['primary'], pad=12)
        ax1.set_xlabel('Attendance rate (%)')
        ax1.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=8, framealpha=0.95, borderaxespad=0)

    ax2 = fig.add_subplot(gs[2, 2:])
    if not df_week.empty:
        wcol  = 'week' if 'week' in df_week.columns else df_week.columns[0]
        x_idx = np.arange(len(df_week))
        ax2.plot(x_idx, df_week['attendance_rate'], marker='o', linewidth=2.2, markersize=5, color=COLORS['accent'])
        ax2.axhline(75, color=COLORS['danger'], linestyle='--', linewidth=1, alpha=0.75)
        ax2.set_ylabel('Attendance rate (%)')
        ax2.set_title('Weekly trend', loc='left', fontsize=12, fontweight='600', color=COLORS['primary'], pad=12)
        n    = len(df_week)
        step = max(1, int(np.ceil(n / 10)))
        ax2.set_xticks(x_idx[::step])
        ax2.set_xticklabels([str(df_week[wcol].iloc[j]) for j in x_idx[::step]], rotation=40, ha='right', fontsize=8)
        ax2.set_xlim(-0.5, n - 0.5)

    ax3 = fig.add_subplot(gs[3, :2])
    if not df_dow.empty:
        gcol = 'day_of_week' if 'day_of_week' in df_dow.columns else df_dow.columns[0]
        sns.barplot(data=df_dow, x=gcol, y='attendance_rate', color=COLORS['success'],
                    alpha=0.88, ax=ax3, edgecolor='white')
        ax3.set_xticks(range(len(df_dow)))
        ax3.set_xticklabels([str(df_dow[gcol].iloc[j])[:3] for j in range(len(df_dow))], rotation=0, ha='center', fontsize=9)
        ax3.set_title('By weekday', loc='left', fontsize=12, fontweight='600', color=COLORS['primary'], pad=12)
        ax3.set_xlabel(''); ax3.set_ylabel('Rate (%)')

    ax4    = fig.add_subplot(gs[3, 2:])
    df_raw = store.merged()
    if not df_raw.empty and 'status' in df_raw.columns:
        sc   = df_raw['status'].value_counts()
        cmap = {'present': COLORS['success'], 'absent': COLORS['danger'],
                'late': COLORS['warning'], 'excused': '#8b5cf6'}
        pie_colors = [cmap.get(str(s).lower(), '#64748b') for s in sc.index]
        total_n    = sc.sum()
        wedges, _  = ax4.pie(
            sc.values, labels=None, autopct=None, colors=pie_colors,
            wedgeprops=dict(width=0.52, edgecolor='white', linewidth=1.2), startangle=90,
        )
        ax4.set_title('Status mix', loc='left', fontsize=12, fontweight='600', color=COLORS['primary'], pad=12)
        legend_labels = [f"{str(i).title():8}  {int(sc[i]):,}  ({100*sc[i]/total_n:.1f}%)" for i in sc.index]
        ax4.legend(wedges, legend_labels, title='Status', loc='center left',
                   bbox_to_anchor=(1.02, 0.5), fontsize=9, framealpha=0.95, borderaxespad=0)
        ax4.set_aspect('equal')

    fig.patch.set_facecolor('#eef2f7')
    if save:
        out = Path('data') / 'dashboards' / 'attendance_dashboard_modern.png'
        out.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out, dpi=160, bbox_inches='tight', facecolor=fig.get_facecolor())
        print(f'Saved static dashboard: {out}')
    return fig


def show_modern_static_dashboard(store, period: str = 'all'):
    build_modern_static_dashboard(store, period=period, save=True)
    plt.show()


print('Dashboard builder ready')

In [ ]:
# CELL 5 — Analysis Agent (ReAct loop, in-memory store)
#
# ExcelsisAgent accepts an optional `checkpointer` parameter.
# When omitted (as here) it falls back to a sync SqliteSaver — fine for notebook use.
# The FastAPI server passes an AsyncSqliteSaver so the SSE streaming path works.

import os
from src.agent import ExcelsisAgent
from src.security import UserContext

CURRENT_USER = UserContext(user_id='analyst')
_agent = ExcelsisAgent(store=store)

print('Agent ready')
print(f"  Model : {os.environ.get('MODEL', 'qwen2.5:14b')}")
print(f'  User  : {CURRENT_USER.user_id}')

In [ ]:
# CELL 5b — Docker Agent Queries (education_db)
# Requires Docker SQL Server running and CELL 2b executed first.

from src.agent import ExcelsisAgent
from src.security import UserContext

_sql_agent = ExcelsisAgent(store=sql_store)
SQL_USER   = UserContext(user_id='analyst')

def ask_sql(question: str) -> str:
    print(f'\nQ: {question}')
    print('─' * 60)
    answer = _sql_agent.ask(question, SQL_USER)
    print(answer)
    return answer


edu_questions = [
    'Give me an overall attendance summary for all students.',
    'Which grade has the lowest attendance rate this year?',
    'List the 10 students with the lowest attendance rates.',
    'What is the attendance trend over the past 6 months?',
    'Which class sections have attendance below 75%?',
]

for q in edu_questions:
    ask_sql(q)

In [ ]:
# CELL 5c — Direct SQL Queries (finance_db + cross-database)
# Requires CELL 2b to have been executed.

import pandas as pd

def run_sql(sql: str, database: str = 'finance_db') -> pd.DataFrame:
    return sql_store._query(sql, database=database)


print('=' * 60)
print('finance_db — Transaction volume by year')
print('=' * 60)
df = run_sql("""
    SELECT YEAR(date) AS fiscal_year, txn_type,
           COUNT(*) AS txn_count, SUM(amount) AS total_amount, AVG(amount) AS avg_amount
    FROM transactions
    GROUP BY YEAR(date), txn_type
    ORDER BY fiscal_year, txn_type
""")
print(df.to_string(index=False))

print('\n' + '=' * 60)
print('finance_db — Top 10 vendors by purchase order value')
print('=' * 60)
df = run_sql("""
    SELECT TOP 10 v.name AS vendor, v.category,
           COUNT(po.po_id) AS order_count, SUM(po.total) AS total_value
    FROM purchase_orders po
    JOIN vendors v ON v.vendor_id = po.vendor_id
    WHERE po.status <> 'cancelled'
    GROUP BY v.name, v.category
    ORDER BY total_value DESC
""")
print(df.to_string(index=False))

print('\n' + '=' * 60)
print('education_db — Attendance rate by class section (lowest first)')
print('=' * 60)
df_edu = sql_store._query("""
    SELECT class_section, grade,
           COUNT(*) AS total_records,
           SUM(CASE WHEN status='present' THEN 1 ELSE 0 END) AS present_count,
           ROUND(100.0 * SUM(CASE WHEN status='present' THEN 1 ELSE 0 END)
                 / NULLIF(COUNT(*), 0), 1) AS attendance_rate
    FROM attendance
    GROUP BY class_section, grade
    ORDER BY attendance_rate ASC
""", database='education_db')
print(df_edu.to_string(index=False))

## Grafana Observability Stack

The stack ships with Prometheus + Grafana pre-configured. Start them alongside SQL Server:

```bash
docker compose -f docker/docker-compose.yml up -d
```

| Service | URL | Credentials |
|---------|-----|-------------|
| **Grafana** | http://localhost:3001 | admin / `$GRAFANA_PASSWORD` (default: `admin`) |
| **Prometheus** | http://localhost:9090 | — |
| **FastAPI metrics** | http://localhost:8000/metrics | — |

Two dashboards are auto-provisioned:
- **Excelsis 360 — API Health** — request rates, p50/p95 latency, error %, rate-limit hits, tool invocations, SQL/RAG cache hit ratios
- **Excelsis 360 — Business Metrics** — attendance rate over time, entities below threshold, top entities by rate

Custom Prometheus metrics emitted by the app:

| Metric | Type | Labels |
|--------|------|--------|
| `agent_tool_invocations_total` | Counter | `tool` |
| `agent_query_duration_seconds` | Histogram | — |
| `agent_query_errors_total` | Counter | — |
| `cache_hits_total` / `cache_misses_total` | Counter | `cache` (`sql` or `rag`) |
| `rate_limit_exceeded_total` | Counter | `path` |

In [ ]:
# CELL 5d — Verify Prometheus metrics endpoint
# Requires the FastAPI backend to be running (bash start.sh or uvicorn api.main:app)

import urllib.request

try:
    with urllib.request.urlopen('http://localhost:8000/metrics', timeout=3) as r:
        lines = r.read().decode().splitlines()
    excelsis_lines = [l for l in lines if l.startswith(('agent_', 'cache_', 'rate_limit_'))]
    print(f'Prometheus endpoint reachable — {len(lines)} metric lines total')
    print('\nCustom Excelsis metrics:')
    for line in excelsis_lines[:30]:
        print(' ', line)
except Exception as e:
    print(f'Backend not reachable: {e}')
    print('Start with: uvicorn api.main:app --host 0.0.0.0 --port 8000')

In [ ]:
# CELL 6 — Generate Sample Data (synthetic, no SQL Server needed)
import random
from datetime import date

store = AttendanceDataStore()  # reset

def generate_sample_data():
    classes = ['10A', '10B', '10C', '11A', '11B', '12A']
    rows, sid = [], 1000
    start = date(2024, 9, 2)

    for cls in classes:
        grade = cls[:2]
        for _ in range(30):
            at_risk = random.random() < 0.15
            day_count = 0
            for offset in range(180):
                if day_count >= 60: break
                d = start + timedelta(days=offset)
                if d.weekday() >= 5: continue
                day_count += 1
                weights = [55, 35, 10] if at_risk else [88, 8, 4]
                status  = random.choices(['present', 'absent', 'late'], weights=weights)[0]
                rows.append({
                    'student_id':   sid,
                    'student_name': f'Student_{sid}',
                    'class':        cls,
                    'grade':        grade,
                    'date':         d.strftime('%Y-%m-%d'),
                    'status':       status,
                })
            sid += 1

    return pd.DataFrame(rows)


df     = generate_sample_data()
result = store.ingest_df(df, 'sample_2024')

print(f'\nData summary:')
print(f'   Records  : {result["rows"]:,}')
print(f'   Students : {df["student_id"].nunique()}')
print(f'   Classes  : {", ".join(df["class"].unique())}')
print(f'   Dates    : {df["date"].min()} → {df["date"].max()}')
print(f'\n{json.dumps(store.summary(), indent=2)}')

In [ ]:
# CELL 7 — Static Dashboard
# Renders a 4-panel figure: by-class bar, weekly trend, weekday bar, status donut.
show_modern_static_dashboard(store)

In [ ]:
# CELL 8 — Interactive Chart
# Edit the query string to change chart type or grouping:
#   'attendance by week line chart', 'attendance by class pie chart', etc.
query    = 'attendance by class bar chart'
p        = parse_attendance_query(query)
df_chart = store.compute_stats(p['group_by'], p['period'])
fig      = build_dashboard(p['chart_type'], df_chart, 'Attendance Rate by Class', p['group_by'])
show_dashboard(fig, save=True)

In [ ]:
# CELL 9 — Agent Queries
# Each call runs a full ReAct loop (model + tools).

def ask(question: str) -> str:
    print(f'\nQ: {question}')
    print('─' * 60)
    answer = _agent.ask(question, CURRENT_USER)
    print(answer)
    return answer


questions = [
    'Give me a summary of overall attendance across all classes.',
    'Which students are at risk? List the 5 with the lowest attendance rate.',
    'Which class has the best attendance and which has the worst?',
]

for q in questions:
    ask(q)